**Packages** **used**

In [ ]:
import pandas
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.layers import Dense
from keras.wrappers.scikit_learn import KerasClassifier
from keras.utils import np_utils
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
...

# Uploading File

In [ ]:
from google.colab import files
uploaded= files.upload()

In [ ]:
# load dataset
dataframe = pandas.read_csv("WQI_Parameter_Scores_1994-2013.csv")
dataset = dataframe.values
dataframe.head()

**This is a multiple class problem and can be solved using Artificial Neural Networks.**
**Based on designations set by CCME(2005) to classify water based on WQI.**
```
Excellent /1    95-100     All measurements are within objectives virtually all of the time
```
```
Good /2          80-94      Conditions rarely depart from natural or desirable levels
```
```
Fair /3          65-79      Conditions sometimes depart from natural or desirable levels
```
```
Marginal /4     45-64      Conditions often depart from natural or desirable levels
```
```
Poor /5          0-44      Conditions usually depart from natural or desirable levels**
```

In [ ]:
def quality_check(num):
    s=0
    if num <=100 and num >=80 :
        s=1
    #elif num <=89 and num >=80:
        #s= 2
    elif num <=79 and num >=65:
        s=2
    elif num <=64 : #and num >=50
        s=3
    #elif num <=49:
     #   s=5
    return s

In [ ]:
dataframe['quality']=dataframe['Overall WQI'].apply(quality_check)

In [ ]:
dataframe

In [ ]:
print(dataframe.loc[dataframe['quality']==4])

In [ ]:
dataset = dataframe.values

In [ ]:
X = dataset[:,4:11].astype(float)
Y = dataset[:,13]


In [ ]:
# encode class values as integers
encoder = LabelEncoder()
encoder.fit(Y)
encoded_Y = encoder.transform(Y)
# convert integers to dummy variables (i.e. one hot encoded)
dummy_y = np_utils.to_categorical(encoded_Y)

## 1 - hot encoding *since multi class problem

In [ ]:
dummy_y

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split( X, dummy_y, test_size=0.2)

In [ ]:
dummy_y.shape

## **Creating an artificial neural network with two hidden layers**
Softmax function has been used to calculate the probability in the output layer.
THe two hidden layers consists 16 hidden units each.The input layer consists of 7 features that are available from the data set -- as discussed earlier.

In [ ]:
def baseline_model():
  classifier2= Sequential()
  classifier2.add(Dense(16,input_dim=7, activation='relu'))
  classifier2.add(Dense(16, activation='relu'))
  classifier2.add(Dense(5, activation='softmax'))
  classifier2.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
  return classifier2  

In [ ]:
estimator = KerasClassifier(build_fn=baseline_model, epochs=1000, batch_size=10, verbose=0)

## K- fold Cross - Validation

In [ ]:
kfold = KFold(n_splits=10, shuffle=True)

In [ ]:
#results = cross_val_score(estimator, X, dummy_y, cv=kfold)
#print("Baseline: %.2f%% (%.2f%%)" % (results.mean()*100, results.std()*100))

In [ ]:
classifier1= Sequential()
classifier1.add(Dense(14,input_dim=7, activation='relu'))
classifier1.add(Dense(10, activation='relu'))
classifier1.add(Dense(10, activation='relu'))
classifier1.add(Dense(3, activation='softmax'))
classifier1.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
classifier1.fit(x_train,y_train, epochs=400, batch_size=4, verbose=1)


In [ ]:
score = classifier1.evaluate(x_test, y_test, verbose = 1) 

print('Test loss:', score[0]) 
print('Test accuracy:', score[1])

**Results : Validation accuracy is 72.8% and test accuracy is 78.4%**